# Emotion Recognition from Dialogue Text

**Project #5 - Emotion Recognition from Text.** Predict the emotion a speaker expresses in each turn of a conversation.

| | |
|---|---|
| **Task** | 7-way single-label classification of a dialogue turn: *neutral, joy, surprise, anger, sadness, disgust, fear* |
| **Data** | MELD (Poria et al., 2019): 13,708 utterances from 1,433 dialogues of the TV series *Friends* |
| **Models** | Majority-class and TF-IDF + Logistic Regression baselines; RoBERTa-base fine-tuned with and without dialogue context |
| **Metrics** | Accuracy, precision, recall, F1 (weighted = standard on MELD, and macro = all emotions equal) |

### 1. Problem understanding

**Who needs this?** Chatbots and virtual assistants that should respond empathetically ("I'm sorry to hear that" vs "Congrats!"), customer-support tools that flag angry customers for escalation, and social-media / mental-health analytics that track emotional tone.

**What makes it hard?**
1. **Conversational text is short and ambiguous.** Half of MELD's utterances have 6 words or fewer. *"What?"* can be surprise, anger or neutral; only the **previous turns** tell which. So the model should read the dialogue context, but only *past* turns (a live chatbot never sees the future).
2. **Heavy class imbalance.** 47% of utterances are neutral; fear and disgust are under 3% each. A model that always says "neutral" already gets ~48% accuracy, so accuracy alone is misleading and we report **macro-F1** as well.
3. **Missing modalities.** MELD annotators watched the video; tone of voice and facial expressions are not in the text, which caps what a text model can reach (published text-only systems score about 0.60-0.67 weighted-F1).

**Constraints:** trained and evaluated on a 4-core CPU without a GPU, so the model must fine-tune in under an hour; inference must be fast enough for an interactive demo (well under a second per message).

In [ ]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.emotion.config import LABELS, RESULTS_DIR, FIGURES_DIR, DEFAULT_TRANSFORMER_DIR, SEED
from src.emotion.data import load_all, add_context, clean_text, model_input
from src.emotion.evaluate import compute_metrics, report, plot_confusion
from src.emotion import baseline as bl

np.random.seed(SEED); torch.manual_seed(SEED)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("torch", torch.__version__, "| project root:", ROOT)

## 2. Dataset

**Why MELD?** The assignment asks for emotions *from dialogues*. MELD is the standard benchmark for emotion recognition in conversation: real multi-party dialogues, a label for every turn, and an official train/dev/test split (so results are comparable with published work). Its seven classes are Ekman's six basic emotions plus neutral, which matches the task description (joy, sadness, anger, ...). The CSVs (1.5 MB) are included in `data/raw/`.

In [ ]:
splits = load_all()
train, dev, test = splits["train"], splits["dev"], splits["test"]
for name, df in splits.items():
    print(f"{name:5s}: {len(df):5d} utterances, {df.dialogue_id.nunique():4d} dialogues")
train.head(8)

In [ ]:
dist = pd.DataFrame({s: df.emotion.value_counts(normalize=True) for s, df in splits.items()}).loc[LABELS]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
(dist * 100).plot.bar(ax=axes[0], rot=0, color=["#4c72b0", "#dd8452", "#55a868"])
axes[0].set_ylabel("% of utterances"); axes[0].set_title("Class distribution (heavily imbalanced)")
lens = train.text.str.split().str.len()
axes[1].hist(lens.clip(upper=40), bins=40, color="#4c72b0")
axes[1].set_xlabel("words per utterance (clipped at 40)"); axes[1].set_title(f"Utterances are short (median {lens.median():.0f} words)")
plt.tight_layout(); plt.savefig(FIGURES_DIR / "data_overview.png", dpi=150); plt.show()
print((dist * 100).round(1).to_string())

**Observations.** Neutral dominates (47%), and fear and disgust are rare (about 270 training examples each). The label distribution is similar across splits, so the test set is representative. Utterances are very short (median 6 words), which motivates using dialogue context.

## 3. Preprocessing pipeline

Implemented in `src/emotion/data.py`:
1. **Unicode / punctuation normalisation:** curly quotes, dashes and ellipses become ASCII, and whitespace is collapsed. Case, punctuation and repeated letters are **kept on purpose**: "WHAT?!" and "what." express different emotions.
2. **No stop-word removal, stemming or lower-casing for the transformer.** RoBERTa's byte-level BPE tokenizer handles any string, and function words ("not", "so", "never") matter for emotion.
3. **Dialogue context.** For every utterance we take the *previous two turns* of the same dialogue (past only). Speaker names are dropped so the model cannot learn character-specific shortcuts that would not transfer to real users.
4. **Model input** = `<s> target utterance </s></s> context </s>`. The target comes first so it is never truncated; overly long contexts are trimmed from the oldest side (max 128 tokens).

In [ ]:
print(repr("I’m so… tired — honestly"), "->", repr(clean_text("I’m so… tired — honestly")))
ctx_test = add_context(test, n_turns=2)
example = ctx_test[ctx_test.dialogue_id == ctx_test.dialogue_id.iloc[40]].head(5)
example[["speaker", "text", "context", "emotion"]]

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(DEFAULT_TRANSFORMER_DIR)
a, b = model_input("What?", "I'm getting married. | To Emily. Next week.")
ids = tok(a, b)["input_ids"]
print(len(ids), "tokens:", tok.convert_ids_to_tokens(ids))

## 4. Baselines

* **Majority class:** always predicts "neutral". This is the floor any useful model must beat.
* **TF-IDF + Logistic Regression** (`src/emotion/baseline.py`): word 1-2-grams (emotional words and phrases such as "oh my god") plus character 2-5-grams (punctuation and spelling such as "!!", "?!", "sooo"), with `class_weight="balanced"` for the imbalance. The regularisation strength `C` is tuned on the **dev** set. This strong, interpretable classical model trains in about a minute.

In [ ]:
from sklearn.dummy import DummyClassifier
majority = DummyClassifier(strategy="most_frequent").fit(train.text, train.label)
m_major = compute_metrics(test.label, majority.predict(test.text))
print(f"Majority: accuracy {m_major['accuracy']:.3f}, weighted-F1 {m_major['weighted_f1']:.3f}, macro-F1 {m_major['macro_f1']:.3f}")

tfidf, best_C = bl.tune_and_train(train, dev)
pred_tfidf = tfidf.predict(test.text)
m_tfidf = compute_metrics(test.label, pred_tfidf)
print(report(test.label, pred_tfidf))

In [ ]:
pd.DataFrame(bl.top_features(tfidf, k=7)).T.rename(columns=lambda i: f"#{i+1}")

The baseline learns sensible cues (*love, great, amazing* → joy; *wow, oh my god, whoa* → surprise; *ugh, ew, disgusting* → disgust; *sorry, died* → sadness). But it cannot handle negation ("not happy"), word order or context, and it struggles with fear and disgust.

## 5. Fine-tuned transformer: RoBERTa-base + dialogue context

**Why RoBERTa?** It is a 125M-parameter transformer pre-trained on 160 GB of English text, including web and conversational-style data, and it is a standard strong encoder for emotion classification. Pre-training already encodes negation, intensity and idioms. Fine-tuning adds a small classification head on the `<s>` token and updates all weights.

**Training setup** (`src/emotion/train_transformer.py`):
* AdamW, learning rate 2e-5, linear warm-up (6%) and decay, batch 16, 4 epochs, weight decay 0.01, gradient clipping 1.0.
* Cross-entropy with **softened class weights** w ∝ 1/√frequency. Full inverse weights over-predict the rare classes; no weights ignores them.
* **Model selection on dev:** the epoch with the best dev weighted-F1 is kept. The test set is used once.
* CPU-friendly: dynamic padding, length-bucketed batches, bfloat16 autocast (Intel AMX), fused AdamW. One run takes about 40 minutes on 4 CPU cores.
* Base weights: `roberta-base` (MIT licence). The development machine could not reach the Hugging Face Hub, so the encoder was taken from spaCy's `en_core_web_trf` package (which bundles RoBERTa-base) and converted with `scripts/convert_spacy_roberta.py`. `scripts/verify_conversion.py` checks that every layer matches the reference implementation to within 2e-5.

Training is done by the script (`python -m src.emotion.train_transformer --base-model roberta-base`). Here we **load the fine-tuned model** shipped in `models/roberta-meld-context/` and evaluate it. Set `TRAIN_TRANSFORMER = True` to retrain from scratch (about 40 minutes with AMX, longer on other CPUs).

In [ ]:
TRAIN_TRANSFORMER = False
if TRAIN_TRANSFORMER:
    import subprocess
    subprocess.run([sys.executable, "-m", "src.emotion.train_transformer", "--base-model", "roberta-base",
                    "--context-turns", "2"], cwd=ROOT, check=True)
    subprocess.run([sys.executable, "scripts/export_model.py", "runs/roberta_context",
                    str(DEFAULT_TRANSFORMER_DIR)], cwd=ROOT, check=True)

hist = pd.DataFrame(json.loads((RESULTS_DIR / "roberta_context_metrics.json").read_text())["history"])
hist_u = pd.DataFrame(json.loads((RESULTS_DIR / "roberta_utterance_metrics.json").read_text())["history"])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.3))
axes[0].plot(hist.epoch, hist.train_loss, "o-", label="with context"); axes[0].plot(hist_u.epoch, hist_u.train_loss, "s--", label="utterance only")
axes[0].set_title("Training loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(hist.epoch, hist.dev_weighted_f1, "o-", label="with context"); axes[1].plot(hist_u.epoch, hist_u.dev_weighted_f1, "s--", label="utterance only")
axes[1].set_title("Dev weighted-F1 (model selection)"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.savefig(FIGURES_DIR / "training_curves.png", dpi=150); plt.show()
hist.round(4)

In [ ]:
import time
from src.emotion.predict import EmotionClassifier
clf = EmotionClassifier()  # models/roberta-meld-context (float32 inference)
contexts = [c.split(" | ") if c else None for c in ctx_test.context]
t0 = time.time()
probs_ctx = clf.predict_proba(list(ctx_test.text), contexts)
print(f"Predicted {len(ctx_test)} test utterances in {time.time() - t0:.0f}s on CPU")
pred_ctx = probs_ctx.argmax(1)
m_ctx = compute_metrics(ctx_test.label, pred_ctx)
print(report(ctx_test.label, pred_ctx))

## 6. Evaluation and comparison

All numbers are on the **held-out test set** (2,610 utterances, never used for training or model selection).
* **Accuracy** is the share of turns labelled correctly.
* **Precision / recall / F1** are computed per class and then averaged. The *weighted* average (by class size) is the standard MELD metric. The *macro* average treats every emotion equally, so it exposes performance on rare classes.

The utterance-only RoBERTa is an **ablation**: the same model and training, but with no context. Its test predictions come from its training run (`results/roberta_utterance_test_probs.npy`).

In [ ]:
probs_utt = np.load(RESULTS_DIR / "roberta_utterance_test_probs.npy")
m_utt = compute_metrics(test.label, probs_utt.argmax(1))
models = {"Majority (always neutral)": m_major, "TF-IDF + LogReg": m_tfidf,
          "RoBERTa (utterance only)": m_utt, "RoBERTa + context (final)": m_ctx}
cols = ["accuracy", "weighted_precision", "weighted_recall", "weighted_f1", "macro_precision", "macro_recall", "macro_f1"]
summary = pd.DataFrame({k: {c: v[c] for c in cols} for k, v in models.items()}).T
summary.round(3)

In [ ]:
# Robustness: the RoBERTa runs were repeated with 3 random seeds (42, 13, 7).
rows = []
for variant in ["utterance", "context"]:
    for seed_suffix in ["", "_seed13", "_seed7"]:
        p = RESULTS_DIR / f"roberta_{variant}{seed_suffix}_metrics.json"
        if p.exists():
            m = json.loads(p.read_text())
            rows.append({"model": f"RoBERTa ({variant})", "seed": m["args"]["seed"], "accuracy": m["accuracy"],
                         "weighted_f1": m["weighted_f1"], "macro_f1": m["macro_f1"]})
seeds = pd.DataFrame(rows)
if len(seeds):
    agg = seeds.groupby("model")[["accuracy", "weighted_f1", "macro_f1"]].agg(["mean", "std", "count"])
    display(agg.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_confusion(test.label, pred_tfidf, "TF-IDF + LogReg", ax=axes[0])
plot_confusion(ctx_test.label, pred_ctx, "RoBERTa + context", ax=axes[1])
plt.tight_layout(); plt.savefig(FIGURES_DIR / "confusion_comparison.png", dpi=150); plt.show()

per_class = pd.DataFrame({k: {l: v["per_class"][l]["f1"] for l in LABELS} for k, v in models.items() if "Majority" not in k})
ax = per_class.plot.bar(figsize=(11, 3.6), rot=0, width=0.8, color=["#bbbbbb", "#8fb3d9", "#2b6cb0"])
ax.set_ylabel("F1 on test"); ax.set_title("Per-class F1"); ax.legend(loc="upper right")
plt.tight_layout(); plt.savefig(FIGURES_DIR / "per_class_f1.png", dpi=150); plt.show()
per_class.round(3)

**Reading the confusion matrices.** Rows are the true emotion and columns the prediction (row-normalised). The most frequent error for every class is **predicting neutral**: weakly expressed emotions look neutral in text. Fear and disgust are the hardest classes, with the fewest examples and frequent confusion with anger and surprise. RoBERTa improves on the baseline for every class.

## 7. Does dialogue context help?

The two RoBERTa models are identical except for the two previous turns given as input. Below are test utterances where the context model is right and the utterance-only model is wrong (and the reverse, for balance).

In [ ]:
cmp = ctx_test[["text", "context", "emotion"]].copy()
cmp["utterance_only"] = [LABELS[i] for i in probs_utt.argmax(1)]
cmp["with_context"] = [LABELS[i] for i in pred_ctx]
fixed = cmp[(cmp.with_context == cmp.emotion) & (cmp.utterance_only != cmp.emotion)]
broken = cmp[(cmp.with_context != cmp.emotion) & (cmp.utterance_only == cmp.emotion)]
print(f"Context fixes {len(fixed)} errors and introduces {len(broken)} new ones (net {len(fixed) - len(broken):+d}).")
short = cmp.text.str.split().str.len() <= 3
analysis = {"fixed": len(fixed), "broken": len(broken)}
for key, name, mask in [("short", "short (<=3 words)", short), ("long", "longer (>3 words)", ~short)]:
    acc_u = (cmp.utterance_only[mask] == cmp.emotion[mask]).mean(); acc_c = (cmp.with_context[mask] == cmp.emotion[mask]).mean()
    analysis.update({f"{key}_n": int(mask.sum()), f"{key}_utt": float(acc_u), f"{key}_ctx": float(acc_c)})
    print(f"  {name:18s} n={mask.sum():4d}  accuracy utterance-only {acc_u:.3f} -> with context {acc_c:.3f}")
(RESULTS_DIR / "context_analysis.json").write_text(json.dumps(analysis, indent=2))
fixed[fixed.text.str.split().str.len() <= 4].head(10)

## 8. Error analysis

In [ ]:
conf = probs_ctx.max(1)
bins = pd.cut(conf, [0, .4, .6, .8, .9, 1.0])
calib = pd.DataFrame({"correct": pred_ctx == ctx_test.label.to_numpy(), "bin": bins}).groupby("bin", observed=True)["correct"].agg(["mean", "count"])
calib.columns = ["accuracy", "n"]
print("Accuracy by model confidence (is the confidence meaningful?):")
display(calib.round(3))

pairs = pd.Series([(LABELS[t], LABELS[p]) for t, p in zip(ctx_test.label, pred_ctx) if t != p]).value_counts().head(6)
print("Most frequent confusions (true -> predicted):"); print(pairs.to_string())

errors = ctx_test.assign(pred=[LABELS[i] for i in pred_ctx], confidence=conf.round(2))
errors = errors[errors.pred != errors.emotion]
errors.sort_values("confidence", ascending=False)[["text", "context", "emotion", "pred", "confidence"]].head(10)

**What goes wrong:**
* **Emotion vs neutral is the main boundary.** Most errors are mild emotions predicted as neutral, or neutral predicted as an emotion. Annotators heard the tone of voice, which the text lacks.
* **Confident mistakes are often label noise or sarcasm.** Many high-confidence "errors" are arguably correct readings of the text, with the gold label driven by delivery: a sarcastic "Oh, great" is labelled anger or disgust but reads as joy.
* **Fear vs surprise vs anger** share surface cues ("Oh my God!", "What?!").
* **The confidence is informative.** Accuracy rises steeply with confidence, so a downstream chatbot could act only on confident predictions and stay neutral otherwise.

## 9. Demo

The interactive demo is `python app.py` (Gradio). The same model is called directly here.

In [ ]:
from src.emotion.predict import BaselineClassifier
base_clf = BaselineClassifier()
examples = [("I can't believe I failed the exam again. I'm so disappointed in myself.", None),
            ("I got the job!! I'm so excited!", None),
            ("What?", ["Rachel, I have something to tell you.", "I'm getting married. To Emily. Next week."]),
            ("What?", ["Hey, I ate your sandwich.", "And I used your toothbrush."]),
            ("Get away from me! There's something moving under the bed!", None),
            ("Oh great, another meeting. Exactly what I needed today.", None)]
rows = []
for text, ctx in examples:
    p = clf.predict(text, context=ctx); top = next(iter(p))
    rows.append({"message": text, "context": " | ".join(ctx) if ctx else "", "RoBERTa+context": f"{top} ({p[top]:.0%})",
                 "TF-IDF baseline": next(iter(base_clf.predict(text)))})
pd.DataFrame(rows)

In [ ]:
# Latency of one prediction (message + 2 context turns) on this CPU
t0 = time.time()
for _ in range(20):
    clf.predict("You're just telling me this now?!", context=["Does this mean you're moving out?", "Well... yes."])
latency = {"ms_per_message": (time.time() - t0) / 20 * 1000, "torch_threads": torch.get_num_threads()}
(RESULTS_DIR / "latency.json").write_text(json.dumps(latency, indent=2))
print(f"{latency['ms_per_message']:.0f} ms per message (CPU, {latency['torch_threads']} threads)")

conversation = ["Guess what? I got the head chef job!", "Oh my God! That's amazing!",
                "I start Monday. I'm kind of terrified, actually.", "You'll be great. But wait, does this mean you're moving out?",
                "Well... yes.", "You're just telling me this now?!"]
pd.DataFrame(clf.predict_conversation(conversation)).round(2)

## 10. Conclusions and limitations

* Fine-tuned **RoBERTa with two turns of dialogue context** is the best model. It clearly beats the TF-IDF baseline on every metric and every class (see the table in Section 6), and context helps most on short, ambiguous replies.
* **Limitations.**
  1. **Text only.** MELD labels were assigned from video, so sarcasm and tone-dependent emotions are the main failure mode, and published text-only results plateau around 0.60-0.67 weighted-F1.
  2. **Rare classes.** Fear and disgust have about 270 training examples each and remain weak (F1 well below the frequent classes).
  3. **One label per turn.** Mixed emotions cannot be expressed.
  4. **Domain.** A 1990s US sitcom; other domains, dialects or code-mixed text (e.g. Hinglish) would need extra data.
  5. **CPU budget.** We used roberta-base with 2 context turns; larger models (roberta-large) and longer context are known to add a few points.
* **Future work:** speaker-aware context, data augmentation or focal loss for rare classes, larger encoders, and adding audio features.